<a href="https://colab.research.google.com/github/BrownParticleAstro/dmtools_notebooks/blob/main/DMToolsPyPiExample_20261104_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DMTOOLS External API Test and Demo - 29 July 2025

Notebook to test library that has been published tp pypi test

In [1]:
## need to disconnect and delete the runtime and rerun the notebook to reinstall this properly
## check that it says it has sucessfully installed
!pip install -i https://test.pypi.org/simple/ dmtools-brown-edu --upgrade

Looking in indexes: https://test.pypi.org/simple/


In [2]:
import json
import os

In [3]:
def load_secrets(env_file_path=".env"):
    """
    Loads secrets from a .env file and retrieves 'EMAIL' and 'API_KEY'.

    Args:
        env_file_path (str): Path to the .env file. Defaults to '.env'.

    Returns:
        dict: A dictionary with 'email' and 'api_key' keys.

    Raises:
        FileNotFoundError: If the .env file does not exist.
        KeyError: If either EMAIL or API_KEY is missing.
    """
    if not os.path.exists(env_file_path):
        raise FileNotFoundError(f"{env_file_path} not found.")

    # Load environment variables from the file
    with open(env_file_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                key_value = line.split('=', 1)
                if len(key_value) == 2:
                    key, value = key_value
                    os.environ[key.strip()] = value.strip()

    email = os.getenv('EMAIL')
    api_key = os.getenv('API_KEY')

    if not email or not api_key:
        missing = []
        if not email:
            missing.append('EMAIL')
        if not api_key:
            missing.append('API_KEY')
        raise KeyError(f"Missing required secrets: {', '.join(missing)}")

    return {"email": email, "api_key": api_key}

In [4]:
cwd = os.getcwd()
print("Current working directory:", cwd)

Current working directory: /content


In [5]:
from dmtools_brown_edu.dmtools_client_package.dmtools_quart import APIClient

In [6]:
## You need to enter your email address into the colab secrets
## see key in left hand bar
## The library cleans the email address provided
## First time you run this you need to give permission to the notebook to access each of the secrets
from google.colab import userdata
email = userdata.get('DM_TOOLS_EMAIL')
api_key = userdata.get('DM_TOOLS')
# print(email)

Create the API Client - this connects to the server and allows simpler interactions.

For less verbose calling, you can set suppress_logs=True

In [7]:
Client = APIClient(email, api_key, suppress_logs=False)

Query using key pair - full match on word
Requirement was to exact word match for the key value pair

Example match to CDMS but not to CDMSSuper

Below returns a list and the results need to be loaded as json

In [8]:
r = Client.read(query="experiment:CDMS")
record_count = len(r)
experiment_list = []
for rc in range(0, record_count):
  r_p = json.loads(r[rc]['properties'])
  r_p_experiment = r_p['experiment']
  experiment_list.append(r_p_experiment)
unique_list = list(set(experiment_list))
print(unique_list)

2026-11-04 12:08:29,341 - dmtools_brown_edu.dmtools_client_package.dmtools_quart - INFO - Calling read data API
INFO:dmtools_brown_edu.dmtools_client_package.dmtools_quart:Calling read data API


[]


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'app.dmtools.brown.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Read a Single record by the ID
Below returns a json variable in a list - the above result from a query needs to be updated to also return a list of json variables

In [11]:
r_id = Client.read(id=19185)
properties_json = r_id[0]['properties']
properties_json

2026-11-04 12:09:59,666 - dmtools_brown_edu.dmtools_client_package.dmtools_quart - INFO - Calling read data API
INFO:dmtools_brown_edu.dmtools_client_package.dmtools_quart:Calling read data API
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'app.dmtools.brown.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'comment': 'DEAP-3600 2019',
 'dateAnnouncement': '2019',
 'dateEnd': None,
 'dateOfficial': None,
 'dateStart': None,
 'defaultColor': 'blue',
 'defaultStyle': 'line',
 'experiment': 'DEAP-3600',
 'greatestHit': 0,
 'label': 'DEAP-3600 2019',
 'label_short': 'DEAP-3600 2019',
 'measurementType': 'Dir',
 'official': 1,
 'open': 0,
 'public': 0,
 'rating': 80,
 'raw': '{[23.12486414 8.00e-43;\n23.81544597 5.66e-43;\n24.51834577 4.10e-43;\n25.40675886 2.91e-43;\n28.46804447 1.17e-43;\n29.43290165 8.47e-44;\n31.20163164 6.21e-44;\n33.08579589 4.47e-44;\n35.18625648 3.24e-44;\n37.52927481 2.37e-44;\n40.0273608 1.81e-44;\n42.69036363 1.46e-44;\n45.52997498 1.20e-44;\n48.55814872 9.93e-45;\n51.78683309 8.49e-45;\n55.22924539 7.46e-45;\n58.90033938 6.58e-45;\n62.81431849 5.98e-45;\n66.98805543 5.49e-45;\n71.43841617 5.11e-45;\n76.18393759 4.82e-45;\n81.24429557 4.57e-45;\n86.64028001 4.38e-45;\n92.39426962 4.23e-45;\n98.52910315 4.17e-45;\n105.0702463 4.19e-45;\n112.0447238 4.26e-45;\n119.48

In [14]:
archived = r_id[0]['record']['archived']
print(archived)

None


In [15]:
archive_via_id = Client.archive(id=19185)

2026-11-04 12:11:33,933 - dmtools_brown_edu.dmtools_client_package.dmtools_quart - INFO - Calling archive data API
INFO:dmtools_brown_edu.dmtools_client_package.dmtools_quart:Calling archive data API
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'app.dmtools.brown.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [16]:
archive_via_id

[{'id': 19185,
  'message': 'Node archived',
  'properties': {'comment': 'DEAP-3600 2019',
   'dateAnnouncement': '2019',
   'dateEnd': None,
   'dateOfficial': None,
   'dateStart': None,
   'defaultColor': 'blue',
   'defaultStyle': 'line',
   'experiment': 'DEAP-3600',
   'greatestHit': 0,
   'label': 'DEAP-3600 2019',
   'label_short': 'DEAP-3600 2019',
   'measurementType': 'Dir',
   'official': 1,
   'open': 0,
   'public': 0,
   'rating': 80,
   'raw': '{[23.12486414 8.00e-43;\n23.81544597 5.66e-43;\n24.51834577 4.10e-43;\n25.40675886 2.91e-43;\n28.46804447 1.17e-43;\n29.43290165 8.47e-44;\n31.20163164 6.21e-44;\n33.08579589 4.47e-44;\n35.18625648 3.24e-44;\n37.52927481 2.37e-44;\n40.0273608 1.81e-44;\n42.69036363 1.46e-44;\n45.52997498 1.20e-44;\n48.55814872 9.93e-45;\n51.78683309 8.49e-45;\n55.22924539 7.46e-45;\n58.90033938 6.58e-45;\n62.81431849 5.98e-45;\n66.98805543 5.49e-45;\n71.43841617 5.11e-45;\n76.18393759 4.82e-45;\n81.24429557 4.57e-45;\n86.64028001 4.38e-45;\n92.39

In [17]:
r_id = Client.read(id=19185)
# r_id

2026-11-04 12:12:51,999 - dmtools_brown_edu.dmtools_client_package.dmtools_quart - INFO - Calling read data API
INFO:dmtools_brown_edu.dmtools_client_package.dmtools_quart:Calling read data API
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'app.dmtools.brown.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [18]:
reverse_archive_via_id = Client.archive_reverse(id=19185)

2026-11-04 12:13:03,610 - dmtools_brown_edu.dmtools_client_package.dmtools_quart - INFO - Calling remove archive data API
INFO:dmtools_brown_edu.dmtools_client_package.dmtools_quart:Calling remove archive data API
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'app.dmtools.brown.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [19]:
reverse_archive_via_id

[{'id': 19185,
  'message': 'Node archive reversed',
  'properties': {'comment': 'DEAP-3600 2019',
   'dateAnnouncement': '2019',
   'dateEnd': None,
   'dateOfficial': None,
   'dateStart': None,
   'defaultColor': 'blue',
   'defaultStyle': 'line',
   'experiment': 'DEAP-3600',
   'greatestHit': 0,
   'label': 'DEAP-3600 2019',
   'label_short': 'DEAP-3600 2019',
   'measurementType': 'Dir',
   'official': 1,
   'open': 0,
   'public': 0,
   'rating': 80,
   'raw': '{[23.12486414 8.00e-43;\n23.81544597 5.66e-43;\n24.51834577 4.10e-43;\n25.40675886 2.91e-43;\n28.46804447 1.17e-43;\n29.43290165 8.47e-44;\n31.20163164 6.21e-44;\n33.08579589 4.47e-44;\n35.18625648 3.24e-44;\n37.52927481 2.37e-44;\n40.0273608 1.81e-44;\n42.69036363 1.46e-44;\n45.52997498 1.20e-44;\n48.55814872 9.93e-45;\n51.78683309 8.49e-45;\n55.22924539 7.46e-45;\n58.90033938 6.58e-45;\n62.81431849 5.98e-45;\n66.98805543 5.49e-45;\n71.43841617 5.11e-45;\n76.18393759 4.82e-45;\n81.24429557 4.57e-45;\n86.64028001 4.38e-45

In [20]:
r_id = Client.read(id=19185)
#r_id

2026-11-04 12:13:19,826 - dmtools_brown_edu.dmtools_client_package.dmtools_quart - INFO - Calling read data API
INFO:dmtools_brown_edu.dmtools_client_package.dmtools_quart:Calling read data API
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'app.dmtools.brown.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [21]:
#new_properties_json = json.loads(r_id)
new_properties = r_id[0]['properties']
type(new_properties)

dict

In [ ]:
new_properties_json = new_properties
#new_properties_json

new_data = Client.create(data=new_properties_json,node_type='Data')

new_data

new_properties_json['rating']=1000

update_data = Client.update(id=15737,updated_data=new_properties_json,node_type='Data')

update_data

r_id = Client.read(id=15737)
r_id

r_id = Client.delete(id=15737)

r_id = Client.read(id=15737)

r_id